## Conceptual definition

At a reliable tracking frame, **h is the probability of losing individual control
through the next selected action**, averaging over a learned pass/carry policy.
It links spatial pressure to possession fragility through explicit choice and
execution components. A carry has a canonical one-second forecast; a pass has
its own completion horizon. Thus h is a **decision-level probability**, not a
per-second rate or the probability of eventual loss by the end of the possession.
A model-based pressure/difficulty profile can be evaluated at arbitrary usable
frames and inspected retrospectively along the observed possession.

The policy and execution models describe an average player in the observed
context. They do not estimate player-specific control skill, body posture or
measured visual orientation. Local geometry/kinematics enter the execution
models; pass options and context enter choice. A high score indicates difficult
modeled alternatives under that policy, not inevitable failure.

## Mathematical definition

Condition all quantities on the current state $s$. Let

* $q=P(\mathrm{pass}\mid s)$; $1-q$ is carry-choice probability.
* $d=P(\mathrm{loss\ within\ 1s}\mid s,\mathrm{carry})$.
* $\pi_j=P(\mathrm{destination}\ j\mid s,\mathrm{pass})$, with $\sum_j\pi_j=1$.
* $p_j=P(\mathrm{pass\ succeeds}\mid s,j)$.

First average pass failure over destinations:

$$f(s)=\sum_j\pi_j(1-p_j)=1-\sum_j\pi_jp_j.$$

Then apply total probability over the two action branches:

$$\boxed{h(s)=(1-q)d+qf(s).}$$

The weighted carry contribution is $(1-q)d$; the weighted pass contribution is
$qf$. These contributions add exactly to h. They differ from the branch risks
d and f themselves. A real case can have high pass risk but low h when passing
is unlikely, or vice versa. All components are evaluated from the state representation at that frame;
the observed next choice and its outcome are not supplied to this formula.
Tracking velocities use the established centered seven-frame smoother
(±0.3s at10Hz), so this is an offline tracking analysis rather than a strictly
causal real-time estimator. No realized future route is supplied as a predictor.

The destination model uses **raw pitch control as an input feature**, while
$p_j$ is the **calibrated pass-success probability** from the physical control
model and its isotonic mapping. The implementation calls
$\sum_j\pi_jp_j$ `rppcf`: this is the project's transition-weighted pass-success
aggregate. Spearman names the underlying control field PPCF; this does not
establish `rPPCF` as a universal term [@spearman2017; @spearman2018beyond].

This composition and its pressure interpretation are **own derivation**.
Discrete hazard terminology follows the event/decision-level idea of a
conditional failure probability [@singer2003]. The internal physical tackle
hazard is a rate and can exceed one; h is a probability in $[0,1]$.

The component vignettes give their assumptions, features, training and references:
[carry loss](../01_ball_drive_risk/01_ball_drive_risk.ipynb),
[pass probability](../02_pass_probability/02_pass_probability.ipynb),
[destination selection](../03_transition_model/03_transition_model.ipynb), and
[action selection](../04_action_selection/04_action_selection.ipynb).


## Operational bands and their interpretation

For a hypothetical constant h over n successive decisions, retention would be
$(1-h)^n$. The number of equal-risk decisions that halves retention is

$$n_{1/2}=\frac{\log(1/2)}{\log(1-h)},\qquad
h_n=1-2^{-1/n}.$$

We use the cutoffs for n=8,4,2. This is an interpretable **project design
choice**, not thresholds learned from outcomes or published clinical-style
pressure categories. The API preserves the established English labels:

| Band | h interval | Constant-risk half-life interpretation |
|---|---|---|
| None | $0\le h<0.082996$ | More than 8 decisions; infinite at h=0 |
| Low | $0.082996\le h<0.159104$ | Between 4 and 8 decisions |
| Medium | $0.159104\le h<0.292893$ | Between 2 and 4 decisions |
| High | $0.292893\le h\le1$ | At most 2 decisions |

The code uses the exact formula, not the rounded displayed numbers. Values
on a threshold enter the higher band. The theoretical n=1 cutoff is h=0.5;
it remains within High and does not define a fifth operational band.
**None means the lowest modeled risk band, not absence of defensive pressure.**
Half-life here is a scale interpretation in **decisions, not seconds**. Passes
have variable duration and actual future states/risks will change; this does
not forecast how many seconds the current player will retain control.

## Transparent numerical example

Suppose q=0.4, d=0.1, destination probabilities are (0.25,0.5,0.25), and pass
success probabilities are (0.7,0.4,0.2). Expected pass failure is f=0.575,
so h=0.6×0.1+0.4×0.575=0.29, in Medium. These are explicitly synthetic inputs.


In [4]:
import numpy as np
from PRESS.utils.hazard import compute_hazard_raw
from PRESS.utils.possession import hazard_band_thresholds, hazard_to_band
from scripts.build_h_vignette_figures import main as rebuild_h_figures

transition = np.array([0.25, 0.50, 0.25])
pass_success = np.array([0.70, 0.40, 0.20])
h, drive_loss, pass_failure, weighted_pass_success = compute_hazard_raw(
    0.1, transition, pass_success, 0.4)
assert np.isclose(h, 0.29) and hazard_to_band(h) == "Medium"
print({"h": h, "carry contribution": 0.6 * drive_loss,
       "pass contribution": 0.4 * pass_failure, "band": hazard_to_band(h)})
print("Exact cutoffs:", hazard_band_thresholds())
rebuild_h_figures()


{'h': 0.29, 'carry contribution': 0.06, 'pass contribution': 0.22999999999999998, 'band': 'Medium'}
Exact cutoffs: {1: 0.5, 2: 0.2928932188134524, 4: 0.1591035847462855, 8: 0.08299595679532878}


![Synthetic branch mixture and exact operational band boundaries.](../../figures/vignette_05_hazard_bands.png)

## Evaluation frames and canonical training actions

The same state formula can be queried at **any reliable tracking frame**.
Canonical action segmentation is a training/evaluation convention, not a
restriction to integer-second prediction times.

* A 0.4s individual control ending with a pass has a pass-choice training row
  at 0.4s. The 0s state can be retained as an additional descriptive h snapshot.
* A 2.4s control ending with a pass has successful canonical carries at 0s and
  1s, plus the pass at 2.4s. A descriptive h profile may also include the 2s
  snapshot; that successful 0.4s tail is not an action-selection training row.
* If that 2.4s control ends in direct loss, the carry starting at 2s is a failed
  training action, even though observed failure occurs after 0.4s. Its target
  remains loss within the canonical one-second forecast.
* A pass before one complete second retains the possession-start context as
  previous action. After at least one complete second of carrying, previous
  action is ball drive. Additional snapshots do not create training labels.

Canonical h benchmarking uses the verified action cohort and matching outer
models. It does not mix descriptive microtail snapshots into labeled actions.
Clearances and fouls are not extra action-choice branches; complete carries
before those endings can contribute canonical states. A foul-interrupted short
tail is not automatically labeled a failed carry. Terminal individual-loss
conventions and the whole-parent quality rules are documented in the
[separate composed h/xLoss benchmark](../05b_composed_risk_benchmark/05b_composed_risk_benchmark.ipynb).

## Three real tracking examples

The examples come from **Brisbane FC–Perth Glory, 21 December 2024**, and use
frozen predictions from the matching held-out fold, not the installed full-data
weights. All selected states belong to the verified canonical cohort. They
are chosen retrospectively to illustrate complementary trajectories; their
selection rules are published with the data and are not an evaluation sample.

* **A, 04:57.7–05:02.4:** h starts at 0.171, peaks at 0.346 and falls to 0.0029
  before a completed pass. At the peak, both execution branches contribute;
  by release, both modeled branch risks are very low. This is a model-based
  risk escape, not proof of a specific tactical or body-orientation mechanism.
* **B, 04:27.1–04:30.4:** h rises from 0.0118 to 0.3498, but the pass succeeds.
  Even the High band leaves substantial probability of successful action.
* **C, 24:49.2–24:52.4:** the terminal pass fails at h=0.3436. It is an example
  of elevated modeled action risk followed by observed failure, not evidence
  that the score alone determines the outcome.

![Real canonical h trajectories and weighted execution-branch contributions. Squares mark observed pass release. Lines connect evaluated snapshots as a visual guide.](../../figures/h_real_case_trajectories.png)

In case A, the peak has carry contribution about0.096 and pass contribution
about0.250. The reduction concerns both branches; it is not merely a shift of
choice probability away from one difficult action. The following code reads
and independently recomposes the published case summaries.


In [6]:
from pathlib import Path
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").is_file())
cases = pd.read_csv(root / "data/examples/hazard/cases.csv", dtype={"match_id": str}, keep_default_na=False)
recomputed = (1 - cases.p_pass) * cases.drive_loss + cases.p_pass * cases.pass_failure
assert np.allclose(recomputed, cases.h, atol=1e-12)
assert np.array_equal(hazard_to_band(cases.h), cases.band)
print(cases[["case", "time_since_start", "action", "p_pass", "drive_loss",
             "pass_failure", "h", "band"]].to_string(index=False))


case  time_since_start     action   p_pass  drive_loss  pass_failure        h   band
   A               0.0 ball_drive 0.289720    0.000260      0.588873 0.170793 Medium
   A               1.0 ball_drive 0.471873    0.010675      0.478794 0.231568 Medium
   A               2.0 ball_drive 0.603692    0.242035      0.414592 0.346206   High
   A               3.0 ball_drive 0.434976    0.081164      0.098052 0.088510    Low
   A               4.7       pass 0.519274    0.002567      0.003206 0.002899   None
   B               0.0 ball_drive 0.203357    0.000240      0.056961 0.011774   None
   B               1.0 ball_drive 0.392803    0.001564      0.092375 0.037235   None
   B               2.0 ball_drive 0.467033    0.028189      0.319292 0.164144 Medium
   B               3.3       pass 0.747304    0.307685      0.364101 0.349845   High
   C               0.0 ball_drive 0.714711    0.119816      0.108703 0.111874    Low
   C               1.0 ball_drive 0.643984    0.093372      0.032

### Case A on the pitch

The snapshots use the package's `plot_frame` on the corresponding observed
tracking frames. The yellow star marks the carrier. Brisbane attacks left;
velocity arrows are scaled drawing aids, not observed future trajectories.
The printed probabilities come from that frame's matching-fold model stack.

![Case A at peak modeled next-action risk.](../../figures/h_real_case_A_peak.png)

![Case A at the later successful pass release, after the modeled risks have fallen.](../../figures/h_real_case_A_exit.png)

[Case values and branch decomposition](../../data/examples/hazard/cases.csv) ·
[selection rules and observed endings](../../data/examples/hazard/case_summary.csv) ·
[provenance](../../data/examples/hazard/metadata.json).

## Use the package on a selected frame

The following executable API example uses **synthetic match data**. Small grids
make it quick and change the model inputs; use the research defaults for
analysis. Initialization does not itself calculate h. Select a reliable frame,
run `compute_frame`, then retrieve its component decomposition. Use
`PRESS.from_skillcorner(...)` from the [match guide](../../usage.qmd) for
processed real matches, with possession/action context and xT supplied as there.


In [8]:
from PRESS import PRESS
from PRESS.datasets import load_demo_match

tracking, possessions, actions, xT_grid = load_demo_match()
model = PRESS(tracking, possessions_data=possessions, actions_data=actions)
possession_id = model.get_possessions().possession_id.iloc[0]
frame_id = int(model.get_frames(possession_id, valid_only=True).frame.iloc[5])
result = model.compute_frame(frame_id, xT_grid,
    ppcf_kwargs={"r_max": 12, "step": 6}, bcos_kwargs={"r_max": 8, "step": 4})
hazard = model.get_hazard_result(frame_id)["Hazard"]
print("hazard:", hazard, "band:", hazard_to_band(hazard))


hazard: 0.012514264786956159 band: None


## From snapshots to possession profiles

h can describe where an individual control becomes difficult, remains under
risk, or escapes risk. For descriptive comparisons, state the checkpoint
sampling and the conditioning population. Longer possessions contribute more
canonical rows; a count of rows in a band is not automatically seconds under
pressure. Comparing teams or players also requires exposure, position and
opponent context; raw band shares do not isolate individual skill.

The [survival/difficulty vignette](../07_survival_curve/07_survival_curve.ipynb)
computes $D_t=-\log(1-h_t)$ and cumulative difficulty. Multiplying
$\prod_t(1-h_t)$ across a supplied retrospective profile is sensitive to its
sampling. **Overlapping one-second forecasts at extra microtail or dense
tracking snapshots must not be treated as independent successive actions.**
Keep a canonical decision sequence for such summaries, or explicitly report a
descriptive index. A denser plot alone must not imply more exposure or loss
probability. There is no conversion of h into a clock-time rate here.

The [typology vignette](../08_possession_typologies/08_possession_typologies.ipynb)
uses band trajectories and observed endings for retrospective labels. The
[team report](../09_league_level_report/09_league_level_report.ipynb) illustrates
aggregation; these are project-defined analysis tools, not externally validated
player or team rankings.

## Evaluation, calibration and scope

The [separate h/xLoss benchmark](../05b_composed_risk_benchmark/05b_composed_risk_benchmark.ipynb)
reports held-out action-loss scores and the pooled calibration curve. h has no
additional post-hoc calibrator. Good calibration of individual components does
not mathematically guarantee calibration of their composition; it is assessed
on the composed predictions. These are development OOF results on ten
repeatedly studied matches, not an untouched external test set.

[Residual risk](../05a_residual_risk/05a_residual_risk.ipynb) completes the model
toward individual terminal xLoss through
$xLoss=h+(1-q)(1-d)c$. It predicts a different target. The raw h pressure profile
and its bands remain usable without that optional continuation model.

The two-action structure excludes shots, and the benchmark is limited to the
agreed outfield, verified-endpoint cohort. Tracking and event-annotation noise,
straight short-horizon extrapolation, unobserved body posture/control skill,
average-player assumptions and limited tactical interactions remain relevant
limitations. The component and BCOS vignettes document these assumptions and
their literature sources. Band thresholds and case-based interpretation are
explicit project choices; their usefulness is not a new benchmark result.
